# Longitudinal Clinical, Dietary, Anthropometric, and Genomic Measurements from Rural Pennsylvania Older Adults Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.03e4-fcq8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id`.

Let's enumerate the record sets and their fields.

In [ ]:
record_sets = [rs['@id'] for rs in metadata.recordSet]
print(f"Available record sets (@id):\n{record_sets}")

# For each record set, print a preview of its records (just 2 rows per set)
for rs_id in record_sets:
    print(f"\n--- Record set: {rs_id} ---")
    records = list(dataset.records(record_set=rs_id))
    if records:
        # Print keys of the first record
        print("Fields (@id):", list(records[0].keys()))
        # Print two example records
        for rec in records[:2]:
            print(rec)
    else:
        print("No records found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, let's load all record sets found and preview their columns.

In [ ]:
# Extract records from each record set
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    # If records are found, load into DataFrame
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Display column names for each record set
for rs_id, df in dataframes.items():
    print(f"\nColumns in record set {rs_id}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: e.g., filtering records, normalizing numeric fields, grouping data.

Choose a record set and numeric field to explore. For demonstration, we'll pick the first available record set and field that is numeric.

In [ ]:
# Helper: Find first record set with numeric field
numeric_types = ['schema:Number', 'schema:Float', 'schema:Integer']

chosen_rs_id = None
numeric_field_id = None

for rs_id in dataframes:
    df = dataframes[rs_id]
    # Try to pick a numeric column
    for col in df.columns:
        # Try to infer numeric via pandas dtype, if not, skip
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            chosen_rs_id = rs_id
            break
    if chosen_rs_id:
        break

if chosen_rs_id and numeric_field_id:
    print(f"Using record set: {chosen_rs_id}, numeric field: {numeric_field_id}")

    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric column
    normalized = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    filtered_df[numeric_field_id + "_normalized"] = normalized
    print("Normalized values:")
    display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Try grouping by another column (pick a string column if exists)
    group_field = None
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped by {group_field}, mean of {numeric_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram for the selected numeric field and a boxplot for grouped statistics (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {chosen_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, and basic EDA of the FAIR^2 dataset with `mlcroissant`. 
* We examined record set and field structure using `@id` references.
* Loaded data into pandas DataFrames.
* Performed sample filtering, normalization, and grouping.
* Visualized distributions and relationships for numeric fields.
Further detailed analysis is possible by referencing the dataset's full Croissant schema and variable dictionaries.